# Pipeline Tag Prediction

## Model A: RoBERTa-base
**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

This notebook implements an improvement model over the baseline using roBERTa-base. It runs in this order:

1. Load split dataframes used in baseline
2. Setup tokenizer
3. Setup HuggingFace Datasets
5. RoBERTa model
6. Setup training configuration and train model
7. Predictions and metric analysis

## 0. Setup

In [1]:
!pip install -q -U evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00


In [2]:
import sys

In [3]:
!{sys.executable} -m pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.6 MB/s eta 0:00:00


In [32]:
import torch
import pandas as pd
import numpy as np
import evaluate
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings, re, ast
from datasets import load_dataset, DatasetDict, Dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

warnings.filterwarnings('ignore')

# Plot defaults
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (12, 5)})

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load DataFrames

Import the train, test, and validation dataframes that were used for the baseline. They underwent an 80/20/20 split. This ensures we are using the same exact dataset across all models to best inform our model and error analysis.

In [6]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, json

DRIVE_DIR_2 = '/content/drive/MyDrive/266-pipeline-tag-prediction'

train_df = pd.read_parquet(f'{DRIVE_DIR_2}/train_df.parquet')
val_df = pd.read_parquet(f'{DRIVE_DIR_2}/val_df.parquet')
test_df = pd.read_parquet(f'{DRIVE_DIR_2}/test_df.parquet')

metadata = json.load(open(f'{DRIVE_DIR_2}/metadata.json'))
label2id, id2label = metadata['label2id'], metadata['id2label']

Mounted at /content/drive


In [7]:
print("Shape of train:", train_df.shape)
print("Shape of validation:", val_df.shape)
print("Shape of test:", test_df.shape)

Shape of train: (112828, 7)
Shape of validation: (14104, 7)
Shape of test: (14104, 7)


## 2. Tokenizer

Setup the tokenizer for the roBERTa model. The context window for the model has a maximum length of 512 tokens. From the data exploration, most of the model cards per model have token counts that fit into the context window, but there is still a portion of cards that exceed it. For those, we simply truncate the text to the first 512 tokens rather than using a sliding window. We explore the benefits of sliding windows as a next step to model improvement.

In [12]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [19]:
# Analyze lengths of model card text to see how much truncation will affect the data

lengths = train_df["text"].str.split().str.len()
print(lengths.describe())
print("\n")
print("Row Index | Text Length")
print(lengths.sort_values(ascending=False).head(50))

count    112828.000000
mean        502.763924
std         872.165612
min           1.000000
25%         115.000000
50%         234.000000
75%         536.000000
max       15633.000000
Name: text, dtype: float64


Row Index | Text Length
74996     15633
4745      15058
9345      14660
9985      14492
103140    14414
55884     13924
62341     13558
17102     13543
38819     13535
48835     13533
78517     13526
86420     13521
107302    13511
77213     13481
59867     13481
110231    13073
55380     12881
75301     12785
99094     12729
23820     12693
81219     12660
45220     12534
100512    12533
59593     12533
15112     12435
40423     12435
49882     12408
110091    12336
91561     12300
97528     12243
18662     12243
36167     12243
72119     11971
39058     11912
47561     11679
73859     11627
82980     11627
61703     11301
49698     11238
111828    11204
37650     11192
15736     11075
80538     11020
17928     10918
66592     10770
29275     10750
55635     10736
90956     1

In [23]:
for t in [500, 1000, 2000, 3000, 5000, 8000]:
    print(f">{t} words:", (lengths > t).sum())
    print(f"<{t} words:", (lengths <= t).sum())
    print()

>500 words: 30160
<500 words: 82668

>1000 words: 13369
<1000 words: 99459

>2000 words: 4687
<2000 words: 108141

>3000 words: 2381
<3000 words: 110447

>5000 words: 866
<5000 words: 111962

>8000 words: 229
<8000 words: 112599



In [24]:
# Load tokenizer from roBERTa-base

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512, # max context window length
    )

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## 3. Convert DataFrames into Hugging Face Dataset objects

Setup the data to be usable by the tokenizer, model, and trainer.

In [25]:
# Convert DataFrames to Hugging Face Datasets

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

In [26]:
# Tokenize each dataset (truncating to the model's max context length)

train_dataset = train_dataset.map(tokenize,
                                  batched=True,
                                  batch_size=32,
                                  remove_columns=["text"], # drop untokenized text col
                                  load_from_cache_file=False, # fresh dataset per run
                                  writer_batch_size=200 # small fix for RAM usage limits
                                  )
val_dataset = val_dataset.map(tokenize,
                              batched=True,
                              batch_size=32,
                              remove_columns=["text"],
                              load_from_cache_file=False,
                              writer_batch_size=200
                              )
test_dataset = test_dataset.map(tokenize,
                                batched=True,
                                batch_size=32,
                                remove_columns=["text"],
                                load_from_cache_file=False,
                                writer_batch_size=200
                                )

Map:   0%|          | 0/112828 [00:00<?, ? examples/s]

Map:   0%|          | 0/14104 [00:00<?, ? examples/s]

Map:   0%|          | 0/14104 [00:00<?, ? examples/s]

In [27]:
# Check dataframes got converted to same amt of dataset examples

print(f"train: {len(train_df)} docs -> {len(train_dataset)} examples")
print(f"val:   {len(val_df)} docs -> {len(val_dataset)} examples")
print(f"test:  {len(test_df)} docs -> {len(test_dataset)} examples")

train: 112828 docs -> 112828 examples
val:   14104 docs -> 14104 examples
test:  14104 docs -> 14104 examples


## 4. RoBERTa Model

Prepare datasets for the model and load the roBERTa model.

In [28]:
# Setup data as PyTorch tensors for model inputs and preserves all columns

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

In [29]:
# Load model

num_labels = len(label2id)
roBERTa_model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_labels)

# Setup metrics - accuracy and F1
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Train

Setup training argument parameters and the trainer.

In [35]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    optim="adamw_torch_fused",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=roBERTa_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Uncomment the block below to check GPU usage while training is running.

In [ ]:
# import subprocess, threading, time

# def log_gpu(interval=5, duration=60):
#     end = time.time() + duration
#     while time.time() < end:
#         out = subprocess.run(
#             ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
#             capture_output=True, text=True
#         )
#         print(out.stdout.strip())
#         time.sleep(interval)

# threading.Thread(target=log_gpu, args=(5, 60)).start()

In [36]:
# Run train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.308590,0.141639,0.955474,0.951971


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3526, training_loss=0.512576824551743, metrics={'train_runtime': 2804.3451, 'train_samples_per_second': 40.233, 'train_steps_per_second': 1.257, 'total_flos': 2.968842648421171e+16, 'train_loss': 0.512576824551743, 'epoch': 1.0})

In [37]:
# Evaluate training
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.308590,0.141639,1,0.955474,0.951971


{'eval_loss': 0.1416390836238861,
 'eval_accuracy': 0.9554736245036869,
 'eval_macro_f1': 0.9519707988208644}

## 6. Predictions and Metric Analysis

In [38]:
test_predictions = trainer.predict(test_dataset)

In [39]:
# Get predicted class per document from logits

preds = test_predictions.predictions.argmax(axis=1)
doc_true_labels = np.array(test_dataset["label"])

In [40]:
# Print check for final prediction counts

print(f"{len(test_df)} documents -> {len(test_dataset)} examples -> {len(preds)} predictions")

14104 documents -> 14104 examples -> 14104 predictions


In [41]:
# Classification Report

target_names = [id2label[str(i)] for i in range(len(id2label))]

print(classification_report(
    doc_true_labels,
    preds,
    target_names=target_names
))

                              precision    recall  f1-score   support

automatic-speech-recognition       0.98      0.97      0.97       617
        image-classification       0.98      0.95      0.96       331
          image-text-to-text       0.84      0.84      0.84      1240
                    robotics       0.95      0.96      0.95       347
         sentence-similarity       0.97      0.95      0.96       343
         text-classification       0.94      0.92      0.93      1033
             text-generation       0.95      0.96      0.96      6689
               text-to-image       1.00      0.99      0.99      2239
        token-classification       0.95      0.93      0.94       362
                 translation       0.98      0.95      0.97       903

                    accuracy                           0.95     14104
                   macro avg       0.95      0.94      0.95     14104
                weighted avg       0.95      0.95      0.95     14104

